# BEKK - Baba-Engle-Kraft-Kroner

**Referencia**: Engle, R. F. & Kroner, K. F. (1995). *Multivariate Simultaneous Generalized ARCH*. Econometric Theory, 11(1), 122-150.

---

O modelo BEKK parametriza **diretamente** a matriz de covariancia condicional $H_t$,
em contraste com CCC/DCC que decompoe em correlacoes e volatilidades.

### Estrutura do modelo BEKK(1,1)

$$H_t = C'C + A' \varepsilon_{t-1} \varepsilon_{t-1}' A + B' H_{t-1} B$$

onde:
- $C$ e uma matriz triangular inferior ($k \times k$)
- $A$ e a matriz de parametros ARCH ($k \times k$)
- $B$ e a matriz de parametros GARCH ($k \times k$)

A formulacao quadratica ($A' \cdot A$, $B' \cdot B$) garante que $H_t$ e **positiva definida**
por construcao.

### Neste notebook

1. O modelo BEKK
2. Parametros e interpretacao
3. BEKK diagonal
4. BEKK vs DCC
5. Covariancias condicionais

> **Nota**: Usamos apenas 2 series do fx_majors por eficiencia computacional,
> pois o BEKK completo tem $O(k^4)$ parametros.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Adicionar utils ao path
sys.path.insert(0, os.path.join("..", "utils"))
from plot_helpers import (
    plot_conditional_covariance,
    plot_correlation_heatmap,
)

from archbox.multivariate import BEKK, DCC

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 100

# Carregar dados fx_majors - usar apenas 2 series para BEKK
data_path = os.path.join("..", "data", "fx_majors.csv")
df_full = pd.read_csv(data_path, parse_dates=["date"], index_col="date")

# Selecionar subset: EUR/USD e GBP/USD (2 series)
cols_subset = ["eurusd", "gbpusd"]
df = df_full[cols_subset]
returns = df.values
labels = [col.upper() for col in df.columns]
dates = df.index

print(f"Dataset (subset): {df.shape[0]} obs x {df.shape[1]} series")
print(f"Series: {labels}")
print(f"\nN parametros BEKK completo (k=2): C={3}, A={4}, B={4} = {11} total")
print(f"N parametros BEKK completo (k=4): C={10}, A={16}, B={16} = {42} total")
print("-> Por isso usamos subset de 2 series!")

## 1. O modelo BEKK

O BEKK completo para $k$ series tem a seguinte estrutura:

$$H_t = C'C + A' \varepsilon_{t-1} \varepsilon_{t-1}' A + B' H_{t-1} B$$

Para $k = 2$:

$$C = \begin{pmatrix} c_{11} & 0 \\ c_{21} & c_{22} \end{pmatrix}, \quad
A = \begin{pmatrix} a_{11} & a_{12} \\ a_{21} & a_{22} \end{pmatrix}, \quad
B = \begin{pmatrix} b_{11} & b_{12} \\ b_{21} & b_{22} \end{pmatrix}$$

**Numero de parametros:**
- $C$: $k(k+1)/2$ (triangular inferior)
- $A$: $k^2$ (completa)
- $B$: $k^2$ (completa)
- **Total**: $k(k+1)/2 + 2k^2$

Para $k=2$: 11 parametros. Para $k=4$: 42 parametros!

In [ ]:
# TODO: Estime BEKK(1,1) com 2 series do fx_majors

# Estimar BEKK completo (full)
model_bekk_full = BEKK(returns, variant="full", univariate_model="GARCH", univariate_order=(1, 1))
results_bekk_full = model_bekk_full.fit(method="mle", disp=True)

print(f"\n{'='*50}")
print("BEKK Completo - Resultados")
print(f"{'='*50}")
print(f"Log-likelihood: {results_bekk_full.loglike:.4f}")
print(f"AIC: {results_bekk_full.aic:.4f}")
print(f"BIC: {results_bekk_full.bic:.4f}")
print(f"N parametros: {len(results_bekk_full.params)}")

## 2. Parametros e interpretacao

As matrizes do BEKK tem interpretacao diferente do CCC/DCC:

- **Matriz $C$**: Define a covariancia de longo prazo (incondicional):
  $\bar{H} = C'C / (1 - A'A - B'B)$ (em sentido matricial)

- **Matriz $A$**: Controla o impacto de **choques** ($\varepsilon_{t-1}$) na covariancia.
  Elementos fora da diagonal permitem **spillover de volatilidade** entre series.

- **Matriz $B$**: Controla a **persistencia** da covariancia condicional.
  Elementos fora da diagonal permitem **spillover de persistencia**.

In [ ]:
# TODO: Extraia e interprete as matrizes de parametros

params = results_bekk_full.params
k = len(labels)

# Reconstruir matrizes a partir dos parametros
# C: triangular inferior k(k+1)/2 params
n_c = k * (k + 1) // 2
n_a = k * k
n_b = k * k

print(f"Total de parametros: {len(params)}")
print(f"Esperado: C={n_c} + A={n_a} + B={n_b} = {n_c + n_a + n_b}")
print(f"\nParametros estimados: {params}")

# Construir C (triangular inferior)
C_mat = np.zeros((k, k))
idx = 0
for i in range(k):
    for j in range(i + 1):
        C_mat[i, j] = params[idx]
        idx += 1

# Construir A e B
A_mat = params[n_c:n_c + n_a].reshape(k, k)
B_mat = params[n_c + n_a:n_c + n_a + n_b].reshape(k, k)

print("\nMatriz C (intercepto):")
print(pd.DataFrame(C_mat, index=labels, columns=labels).round(6))

print("\nMatriz A (ARCH - impacto de choques):")
print(pd.DataFrame(A_mat, index=labels, columns=labels).round(6))

print("\nMatriz B (GARCH - persistencia):")
print(pd.DataFrame(B_mat, index=labels, columns=labels).round(6))

# Covariancia incondicional
print("\nC'C (contribuicao do intercepto):")
print(pd.DataFrame(C_mat.T @ C_mat, index=labels, columns=labels).round(8))

## 3. BEKK diagonal

O **BEKK diagonal** impoe que $A$ e $B$ sejam matrizes diagonais:

$$A = \begin{pmatrix} a_{11} & 0 \\ 0 & a_{22} \end{pmatrix}, \quad
B = \begin{pmatrix} b_{11} & 0 \\ 0 & b_{22} \end{pmatrix}$$

Isso elimina os **spillovers de volatilidade** entre series, reduzindo o numero
de parametros de $k(k+1)/2 + 2k^2$ para $k(k+1)/2 + 2k$.

Para $k=2$: de 11 para 7 parametros.

In [ ]:
# TODO: Estime BEKK diagonal e compare com BEKK completo

# Estimar BEKK diagonal
model_bekk_diag = BEKK(returns, variant="diagonal", univariate_model="GARCH", univariate_order=(1, 1))
results_bekk_diag = model_bekk_diag.fit(method="mle", disp=True)

print(f"\n{'='*60}")
print("Comparacao BEKK Completo vs Diagonal")
print(f"{'='*60}")

comp = pd.DataFrame({
    "Modelo": ["BEKK Full", "BEKK Diagonal"],
    "Log-Likelihood": [results_bekk_full.loglike, results_bekk_diag.loglike],
    "AIC": [results_bekk_full.aic, results_bekk_diag.aic],
    "BIC": [results_bekk_full.bic, results_bekk_diag.bic],
    "N Params": [len(results_bekk_full.params), len(results_bekk_diag.params)],
}).set_index("Modelo")

print(comp.to_string())

# Teste de razao de verossimilhanca (LR test)
from scipy import stats

lr_stat = 2 * (results_bekk_full.loglike - results_bekk_diag.loglike)
df_diff = len(results_bekk_full.params) - len(results_bekk_diag.params)
if df_diff > 0:
    p_value = 1 - stats.chi2.cdf(lr_stat, df_diff)
    print(f"\nTeste LR: estatistica={lr_stat:.4f}, gl={df_diff}, p-valor={p_value:.4f}")
    print(f"{'Rejeita' if p_value < 0.05 else 'Nao rejeita'} H0 (diagonal) a 5%")

## 4. BEKK vs DCC

**Tradeoffs** entre BEKK e DCC:

| Aspecto | BEKK | DCC |
|---------|------|-----|
| Parametrizacao | Direta em $H_t$ | Via correlacoes $R_t$ |
| Pos. definida | Garantida | Garantida |
| N parametros | $O(k^4)$ completo | $O(k)$ + 2 |
| Spillovers | Modelados | Nao modelados |
| Escalabilidade | Ate ~5 series | Centenas de series |
| Estimacao | MLE conjunta | Duas etapas |

In [ ]:
# TODO: Compare BEKK e DCC no mesmo dataset

# Estimar DCC com as mesmas 2 series
model_dcc = DCC(returns, univariate_model="GARCH", univariate_order=(1, 1))
results_dcc = model_dcc.fit(method="two_step", disp=False)

# Tabela comparativa completa
comp_all = pd.DataFrame({
    "Modelo": ["BEKK Full", "BEKK Diagonal", "DCC"],
    "Log-Likelihood": [
        results_bekk_full.loglike, results_bekk_diag.loglike, results_dcc.loglike
    ],
    "AIC": [results_bekk_full.aic, results_bekk_diag.aic, results_dcc.aic],
    "BIC": [results_bekk_full.bic, results_bekk_diag.bic, results_dcc.bic],
    "N Params": [
        len(results_bekk_full.params), len(results_bekk_diag.params), len(results_dcc.params)
    ],
}).set_index("Modelo")

print("Comparacao BEKK vs DCC (2 series):")
print(comp_all.to_string())

# Comparar correlacoes dinamicas
R_bekk = results_bekk_full.dynamic_correlation  # (T, 2, 2)
R_dcc = results_dcc.dynamic_correlation

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(dates, R_bekk[:, 0, 1], label="BEKK Full", linewidth=0.8, alpha=0.8)
ax.plot(dates, R_dcc[:, 0, 1], label="DCC", linewidth=0.8, alpha=0.8)
ax.set_title(f"Correlacao Condicional: {labels[0]}-{labels[1]}")
ax.set_ylabel("Correlacao")
ax.set_xlabel("Data")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 5. Covariancias condicionais

O BEKK modela diretamente a covariancia condicional $H_t$, permitindo visualizar
tanto as **variancias** (diagonal) quanto as **covariancias** (fora da diagonal)
ao longo do tempo.

In [ ]:
# TODO: Plote as covariancias condicionais do BEKK

H_bekk = results_bekk_full.dynamic_covariance  # (T, 2, 2)

# Preparar dicionario de covariancias
cov_dict = {
    f"Var({labels[0]})": H_bekk[:, 0, 0],
    f"Var({labels[1]})": H_bekk[:, 1, 1],
    f"Cov({labels[0]},{labels[1]})": H_bekk[:, 0, 1],
}

plot_conditional_covariance(dates, cov_dict, title="BEKK - Covariancias Condicionais")
plt.show()

# Heatmaps em datas selecionadas
sample_indices = [0, len(dates)//4, len(dates)//2, 3*len(dates)//4, -1]

fig, axes = plt.subplots(1, len(sample_indices), figsize=(20, 4))
for idx, t in enumerate(sample_indices):
    plot_correlation_heatmap(
        H_bekk[t], labels,
        title=f"$H_t$ ({dates[t].strftime('%Y-%m-%d')})",
        ax=axes[idx], vmin=None, vmax=None
    )

fig.suptitle("BEKK - Matriz de Covariancia em Datas Selecionadas", fontsize=14, y=1.05)
fig.tight_layout()
plt.show()

# Estatisticas
print("Estatisticas das covariancias condicionais:")
for name, series in cov_dict.items():
    print(f"  {name}: media={np.mean(series):.6f}, std={np.std(series):.6f}")